# 10 — Bronze: Receita Federal — Simples Nacional

## Purpose
Ingests Receita Federal Simples Nacional data into the Bronze layer.
Single file (~48M rows) with Simples/MEI optant flags and dates.

## Input
- `data/raw/receita_federal/{YYYY-MM}/Simples.zip` — single file (~291 MB compressed)
- ~48M rows

## Output
- `data/bronze/rf_simples/_year_month={YYYY-MM}/data.parquet`
- Schema: 7 source columns (all STRING) + 5 technical columns

## Steps
1. Imports and configuration
2. Schema definition
3. Process file
4. Validation
5. Ops registration

## Notes
- No headers — columns identified by position per official RF layout
- Sentinel `00000000` for missing dates — Silver converts to NULL
- 51.4% of `data_exclusao_simples` is `00000000` (company still optant)
- Temp file pattern: ZIP extracted to disk → DuckDB reads CSV → temp deleted
- Encoding: latin-1 (RF standard) with ignore_errors=true as safety net
- Idempotent — partition overwritten on re-run
- ADR-002: all source columns as STRING


## Step 1 — Imports and configuration

In [ ]:
import json
import os
import sys
import tempfile
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import get_project_root, to_sql_path
from duckdb_utils import get_connection
from validation import CheckSuite
from bootstrap_log import append_log, make_entry
from pipeline import check_landing_gate

PROJECT_ROOT = get_project_root()

RF_ROOT     = PROJECT_ROOT / "data" / "raw"    / "receita_federal"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze" / "rf_simples"
DRIFT_LOG   = PROJECT_ROOT / "db"  / "schema_drift_log.json"
LOG_PATH    = PROJECT_ROOT / "db"  / "bootstrap_log.json"

check_landing_gate(PROJECT_ROOT)

available_months = sorted([d.name for d in RF_ROOT.iterdir() if d.is_dir()])
if not available_months:
    raise FileNotFoundError(
        f"No month directories found in {RF_ROOT}\n"
        "Run 01_bootstrap_receita_federal.py first."
    )

SAMPLE_MONTH = available_months[-1]
SAMPLE_DIR   = RF_ROOT / SAMPLE_MONTH

SOURCE_ID  = "receita_federal"
BATCH_ID   = str(uuid.uuid4())
STARTED_AT = datetime.now(timezone.utc).isoformat()

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"SAMPLE_MONTH : {SAMPLE_MONTH}")
print(f"BATCH_ID     : {BATCH_ID}")
print(f"STARTED_AT   : {STARTED_AT}")


## Step 2 — Schema definition

In [ ]:
# ---------------------------------------------------------------------------
# Simples layout — 7 columns per official RF documentation
# Reference: https://arquivos.receitafederal.gov.br (layout PDF)
# Columns identified by position — CSV has NO headers
# ---------------------------------------------------------------------------
SIMPLES_COLS = [
    "cnpj_basico",            # 0 — 8-digit CNPJ root (FK rf_empresas)
    "opcao_simples",          # 1 — S/N — Simples Nacional optant
    "data_opcao_simples",     # 2 — YYYYMMDD or sentinel 00000000
    "data_exclusao_simples",  # 3 — YYYYMMDD or 00000000 (51.4% = still optant)
    "opcao_mei",              # 4 — S/N — MEI optant
    "data_opcao_mei",         # 5 — YYYYMMDD or sentinel 00000000
    "data_exclusao_mei",      # 6 — YYYYMMDD or 00000000 (still MEI)
]

EXPECTED_COL_COUNT = len(SIMPLES_COLS)  # 7 — drift = != 7 columns

TECHNICAL_COLUMNS = [
    "_source_id",
    "_batch_id",
    "_ingested_at",
    "_source_file",
    "_year_month",
]

ALL_COLUMNS = SIMPLES_COLS + TECHNICAL_COLUMNS

# Simples has 7 columns — DuckDB uses column0..column6 (no zero-padding needed for < 10)
COL_ALIASES = ", ".join(f"column{i} AS {c}" for i, c in enumerate(SIMPLES_COLS))

print(f"Source columns    : {len(SIMPLES_COLS)}")
print(f"Technical columns : {len(TECHNICAL_COLUMNS)}")
print(f"Total columns     : {len(ALL_COLUMNS)}")
print()
print("[NOTE] Sentinel 00000000 in date fields = missing date, NOT a real date")
print("       Silver rule: CASE WHEN col = '00000000' THEN NULL")
print("                         ELSE TRY_STRPTIME(col, '%Y%m%d')::DATE END")


## Step 3 — Process file

In [ ]:
def _log_drift_event(actual_cols: int, expected_cols: int) -> None:
    """
    Append a schema drift event to schema_drift_log.json.

    For positional CSV sources, drift = unexpected column count.

    Parameters
    ----------
    actual_cols   : number of columns found in the file
    expected_cols : expected column count (EXPECTED_COL_COUNT)
    """
    event = {
        "batch_id"      : BATCH_ID,
        "source_id"     : SOURCE_ID,
        "object"        : "bronze_rf_simples",
        "event_type"    : "SCHEMA_DRIFT",
        "severity"      : "WARN",
        "action"        : "CONTINUE",
        "expected_cols" : expected_cols,
        "actual_cols"   : actual_cols,
        "logged_at"     : datetime.now(timezone.utc).isoformat(),
    }
    existing = []
    if DRIFT_LOG.exists():
        with open(DRIFT_LOG, "r", encoding="utf-8") as f:
            try:
                existing = json.load(f)
            except json.JSONDecodeError:
                existing = []
    existing.append(event)
    DRIFT_LOG.parent.mkdir(parents=True, exist_ok=True)
    with open(DRIFT_LOG, "w", encoding="utf-8") as f:
        json.dump(existing, f, ensure_ascii=False, indent=2)


# ---------------------------------------------------------------------------
# Process Simples.zip — single file, temp extraction pattern
# ---------------------------------------------------------------------------
zip_path = SAMPLE_DIR / "Simples.zip"

if not zip_path.exists():
    raise FileNotFoundError(
        f"Simples.zip not found in {SAMPLE_DIR}\n"
        "Run 01_bootstrap_receita_federal.py first."
    )

BRONZE_ROOT.mkdir(parents=True, exist_ok=True)
partition_dir = BRONZE_ROOT / f"_year_month={SAMPLE_MONTH}"
partition_dir.mkdir(parents=True, exist_ok=True)
output_file  = partition_dir / "data.parquet"
output_path  = to_sql_path(output_file)
source_sql   = to_sql_path(zip_path)

has_drift     = False
total_records = 0
error_count   = 0

print(f"Source : {zip_path.name} ({zip_path.stat().st_size / 1e6:.1f} MB compressed)")
print(f"Output : {output_file}")
print()

tmp_path = None
try:
    # Extract CSV to temp file — DuckDB reads in streaming chunks (never loads 48M rows into RAM)
    print("Extracting to temp file...")
    with zipfile.ZipFile(zip_path) as zf:
        inner_name = zf.namelist()[0]
        with tempfile.NamedTemporaryFile(suffix=".csv", delete=False, mode="wb") as tmp:
            tmp.write(zf.read(inner_name))
            tmp_path = tmp.name

    tmp_posix = to_sql_path(tmp_path)  # forward slashes for DuckDB SQL
    print(f"Temp file: {tmp_path} ({Path(tmp_path).stat().st_size / 1e6:.1f} MB extracted)")
    print()

    con = get_connection()  # in-memory — Parquet I/O does not require persistent DB

    # Schema drift check — sample first rows to count columns
    sample_desc = con.execute(f"""
        SELECT * FROM read_csv('{tmp_posix}',
            sep=';', header=false,
            encoding='latin-1',
            ignore_errors=true,
            sample_size=100
        ) LIMIT 1
    """).description
    actual_cols = len(sample_desc)

    if actual_cols != EXPECTED_COL_COUNT:
        has_drift = True
        _log_drift_event(actual_cols, EXPECTED_COL_COUNT)
        print(f"[DRIFT] Expected {EXPECTED_COL_COUNT} cols, got {actual_cols} — logged to schema_drift_log.json")

    print("Writing Bronze Parquet...")
    con.execute(f"""
        COPY (
            SELECT
                {COL_ALIASES},
                '{SOURCE_ID}'    AS _source_id,
                '{BATCH_ID}'     AS _batch_id,
                CURRENT_TIMESTAMP AS _ingested_at,
                '{source_sql}'   AS _source_file,
                '{SAMPLE_MONTH}' AS _year_month
            FROM read_csv('{tmp_posix}',
                sep=';', header=false,
                encoding='latin-1',
                ignore_errors=true
            )
        ) TO '{output_path}'
        (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)

    total_records = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{output_path}')"
    ).fetchone()[0]
    con.close()

    print(f"OK Simples — {total_records:,} records written")

except Exception as e:
    error_count = 1
    print(f"ERR {e}")
    raise

finally:
    # Always delete temp file — even if an exception occurs
    if tmp_path and Path(tmp_path).exists():
        os.unlink(tmp_path)
        print(f"Temp file deleted: {tmp_path}")


## Step 4 — Validation

In [ ]:
suite   = CheckSuite("bronze_rf_simples")
con_val = get_connection()

# Check 1 — no processing errors
suite.add("No processing errors", error_count == 0, f"{error_count} error(s)")

# Check 2 — output Parquet exists
suite.add("Output Parquet file exists", output_file.exists(), str(output_file))

# Check 3 — total records > 0
suite.add("Total records > 0", total_records > 0, f"{total_records:,}")

# Check 4 — record count from Parquet matches in-memory count
bronze_total = con_val.execute(
    f"SELECT COUNT(*) FROM read_parquet('{output_path}')"
).fetchone()[0]
suite.add(
    "Record count matches",
    bronze_total == total_records,
    f"parquet={bronze_total:,}  counter={total_records:,}",
)

# Check 5 — column count correct
actual_cols = con_val.execute(
    f"SELECT COUNT(*) FROM (DESCRIBE SELECT * FROM read_parquet('{output_path}') LIMIT 0)"
).fetchone()[0]
suite.add(
    "Column count correct",
    actual_cols == len(ALL_COLUMNS),
    f"found={actual_cols}  expected={len(ALL_COLUMNS)}",
)

# Check 6 — all technical columns present
existing_cols = {
    row[0] for row in con_val.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{output_path}') LIMIT 0"
    ).fetchall()
}
missing_tech = [c for c in TECHNICAL_COLUMNS if c not in existing_cols]
suite.add(
    "All technical columns present",
    not missing_tech,
    f"missing: {missing_tech}" if missing_tech else "all present",
)

# Check 7 — sentinel 00000000 preserved as STRING
sentinel_count = con_val.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{output_path}')
    WHERE data_exclusao_simples = '00000000'
""").fetchone()[0]
suite.add(
    "Sentinel 00000000 preserved as STRING",
    sentinel_count > 0,
    f"{sentinel_count:,} records with sentinel (expected ~51% of total)",
)

# Check 8 — opcao_simples and opcao_mei only S or N
invalid_flags = con_val.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{output_path}')
    WHERE opcao_simples NOT IN ('S', 'N') OR opcao_mei NOT IN ('S', 'N')
""").fetchone()[0]
suite.add(
    "opcao_simples and opcao_mei only S or N",
    invalid_flags == 0,
    f"{invalid_flags:,} invalid flag values",
)

# Check 9 — schema drift monitored (always PASS)
suite.add(
    "Schema drift monitored",
    True,
    "drift detected — check schema_drift_log.json" if has_drift else "none detected",
)

con_val.close()  # close before potential raise
suite.report()
suite.assert_all_pass()


## Step 5 — Ops registration

In [ ]:
bytes_written = output_file.stat().st_size if output_file.exists() else 0

entry = make_entry(
    source_id     = SOURCE_ID,
    period        = SAMPLE_MONTH,
    status        = "SUCCESS" if error_count == 0 else "ERROR",
    records       = total_records,
    bytes_written = bytes_written,
    started_at    = STARTED_AT,
    finished_at   = datetime.now(timezone.utc).isoformat(),
    batch_id         = BATCH_ID,
    layer            = "bronze",
    object           = "rf_simples",
    files            = 1,
    has_rescued_data = has_drift,
    drift_months     = 1 if has_drift else 0,
)

append_log(LOG_PATH, entry)

print(f"Batch registered   : {BATCH_ID}")
print(f"Status             : {entry['status']}")
print(f"Records            : {total_records:,}")
print(f"Bytes written      : {bytes_written / (1024*1024):.1f} MB")
print(f"Has rescued data   : {has_drift}")
print(f"Log                : {LOG_PATH}")
